In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

import pandas as pd

sys.path.append("..")

from predql.converter import SConverter, TConverter

In [3]:
from relbench.datasets import get_dataset, get_dataset_names
from relbench.tasks import get_task_names

get_dataset_names()

['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-trial']

In [4]:
def merge_dataframes(df_rb, df_predql):
    merged = pd.merge(
    df_rb,       
    df_predql,   
    on=['fk', 'timestamp'], 
    how='outer',       
    suffixes=('_rb', '_predql'), 
    indicator=True    
    )

    print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
    print(f"Only in PredQL:\n {merged[merged['_merge'] == 'right_only']}")
    print(f"In both:\n {merged[merged['_merge'] == 'both']}")

In [5]:
def process_df_rb(df_rb, fk, timestamp, label):
    renamed_df_rb = df_rb.rename(columns={fk        : 'fk',
                                          timestamp : 'timestamp',
                                          label     : 'label'})
    df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])

    return df_rb

# F1 Database

In [6]:
dataset_f1 = get_dataset(name="rel-f1", download=True)

db_f1 = dataset_f1.get_db()

timestamps_f1 = pd.date_range(start="1960-05-13", end="2005-01-01 ", freq="ME")

sconverter_f1 = SConverter(db_f1)
tconverter_f1 = TConverter(db_f1, timestamps_f1)

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.03 seconds.


In [7]:
get_task_names("rel-f1")

['driver-position', 'driver-dnf', 'driver-top3']

## Entity Classification Tasks

### driver-dnf

Task Description: For each driver predict the if they will DNF (did not finish) a race in the next 1 month. 

In [8]:
# RelBench table

from relbench.tasks.f1 import DriverDNFTask 

task_table_rb = DriverDNFTask(dataset_f1).make_table(db=db_f1, timestamps=timestamps_f1)
print(task_table_rb)

Table(df=
           date  driverId  did_not_finish
0    1999-05-31        14               1
1    1999-06-30        67               1
2    1999-06-30        55               0
3    1999-07-31        22               1
4    1999-08-31        22               0
...         ...       ...             ...
9651 1969-05-31       345               0
9652 1969-06-30       305               1
9653 1969-06-30       346               1
9654 1969-07-31       349               1
9655 1969-08-31       350               1

[9656 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)


In [ ]:
# PredQL table

predql_query = """
    PREDICT MAX(results.statusId, 1, 31, DAYS) != 1
    FOR EACH drivers.driverId
    WHERE MAX(results.statusId, 1, 31, DAYS) IS NOT NULL;
"""

task_table_predql = tconverter_f1.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        for_each.timestamp,
    FROM
        drivers parent
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            MAX(aggr_tbl.statusId)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            results aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        AND
            aggr_tbl.date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.date < time.timestamp + INTERVAL '31 DAY'
        GROUP BY
            time.timestamp, parent.driverId
        ------AGGREGATION_END-

In [10]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 45]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 45]
print(filtered_df_predql)

           date  driverId  did_not_finish
6553 2004-02-29        45               1
53   2004-03-31        45               1
5988 2004-04-30        45               1
9109 2004-05-31        45               1
4788 2004-06-30        45               1
7231 2004-07-31        45               1
5392 2004-08-31        45               1
70   2004-09-30        45               1
      fk  timestamp  label
9508  45 2004-02-29   True
9528  45 2004-03-31   True
9548  45 2004-04-30   True
9569  45 2004-05-31   True
9589  45 2004-06-30   True
9610  45 2004-07-31   True
9633  45 2004-08-31   True
9654  45 2004-09-30   True


In [11]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 40]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 40]
print(filtered_df_predql)

           date  driverId  did_not_finish
5454 1999-02-28        40               1
6056 1999-03-31        40               1
6496 1999-05-31        40               1
4150 1999-06-30        40               1
7835 1999-07-31        40               1
7840 1999-08-31        40               1
5344 1999-09-30        40               1
7845 2000-02-29        40               1
8431 2000-03-31        40               1
4160 2000-04-30        40               1
4161 2000-05-31        40               1
4166 2000-06-30        40               1
1248 2000-07-31        40               1
13   2000-08-31        40               0
1287 2000-09-30        40               1
9065 2001-05-31        40               1
4173 2001-06-30        40               1
7230 2004-07-31        40               1
5393 2004-08-31        40               1
71   2004-09-30        40               1
      fk  timestamp  label
8629  40 1999-02-28   True
8651  40 1999-03-31   True
8695  40 1999-05-31   True
8717  40 1

In [12]:
df_rb = process_df_rb(df_rb, fk='driverId', timestamp='date', label='did_not_finish')
df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

      timestamp   fk  label
6400 1960-05-31  288      1
9561 1960-05-31  346      1
542  1960-05-31  355      0
7061 1960-05-31  359      1
8946 1960-05-31  363      1
...         ...  ...    ...
9111 2004-09-30   34      1
71   2004-09-30   40      1
69   2004-09-30   43      1
70   2004-09-30   45      1
1303 2004-09-30   46      1

[9656 rows x 3 columns]
       fk  timestamp  label
0     288 1960-05-31      1
1     346 1960-05-31      1
2     355 1960-05-31      0
3     359 1960-05-31      1
4     363 1960-05-31      1
...   ...        ...    ...
9651   34 2004-09-30      1
9652   40 2004-09-30      1
9653   43 2004-09-30      1
9654   45 2004-09-30      1
9655   46 2004-09-30      1

[9656 rows x 3 columns]


In [13]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
       timestamp   fk  label_rb  label_predql _merge
0    2000-02-29    1         1             1   both
1    2000-03-31    1         1             1   both
2    2000-04-30    1         1             1   both
3    2000-05-31    1         1             1   both
4    2000-06-30    1         1             1   both
...         ...  ...       ...           ...    ...
9651 1960-08-31  544         1             1   both
9652 1960-08-31  545         1             1   both
9653 1960-08-31  546         1             1   both
9654 1960-08-31  547         1             1   both
9655 1960-10-31  548         1             1   both

[9656 rows x 5 columns]


### driver-top3

Task Description: For each driver predict if they will qualify in the top-3 for a race in the next 1 month. 

In [14]:
# RelBench table

from relbench.tasks.f1 import DriverTop3Task

task_table_rb = DriverTop3Task(dataset_f1).make_table(db=db_f1, timestamps=timestamps_f1)
print(task_table_rb)

Table(df=
           date  driverId  qualifying
0    2004-06-30        12           0
1    2004-06-30        20           0
2    2004-07-31        46           0
3    2004-07-31        13           0
4    2004-09-30         1           0
...         ...       ...         ...
1285 2003-07-31         3           1
1286 2003-08-31         3           0
1287 2003-09-30        21           1
1288 2004-02-29         3           0
1289 2004-05-31         1           0

[1290 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)


In [15]:
# PredQL table

predql_query = """
    PREDICT MIN(qualifying.position, 1, 31, DAYS) <= 3
    FOR EACH drivers.driverId
    WHERE MIN(qualifying.position, 1, 31, DAYS) IS NOT NULL;
"""

task_table_predql = tconverter_f1.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        for_each.timestamp,
    FROM
        drivers parent
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            MIN(aggr_tbl.position)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            qualifying aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        AND
            aggr_tbl.date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.date < time.timestamp + INTERVAL '31 DAY'
        GROUP BY
            time.timestamp, parent.driverId
        ------AGGREGATION_E

In [16]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 45]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 45]
print(filtered_df_predql)

           date  driverId  qualifying
1054 2004-02-29        45           0
200  2004-03-31        45           0
921  2004-04-30        45           0
302  2004-05-31        45           0
749  2004-06-30        45           0
1137 2004-07-31        45           0
820  2004-08-31        45           0
165  2004-09-30        45           0
      fk  timestamp  label
1142  45 2004-02-29  False
1162  45 2004-03-31  False
1182  45 2004-04-30  False
1203  45 2004-05-31  False
1223  45 2004-06-30  False
1244  45 2004-07-31  False
1267  45 2004-08-31  False
1288  45 2004-09-30  False


In [17]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 40]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 40]
print(filtered_df_predql)

           date  driverId  qualifying
864  1999-02-28        40           0
539  2000-02-29        40           0
716  2000-03-31        40           0
646  2000-09-30        40           0
1138 2004-07-31        40           0
818  2004-08-31        40           0
166  2004-09-30        40           0
      fk  timestamp  label
775   40 1999-02-28  False
842   40 2000-02-29  False
864   40 2000-03-31  False
886   40 2000-09-30  False
1240  40 2004-07-31  False
1263  40 2004-08-31  False
1286  40 2004-09-30  False


In [18]:
df_rb = process_df_rb(df_rb, fk='driverId', timestamp='date', label='qualifying')
df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

     timestamp  fk  label
6   1994-02-28  21      0
400 1994-02-28  29      1
312 1994-02-28  43      0
582 1994-02-28  48      0
401 1994-02-28  49      0
..         ...  ..    ...
239 2004-09-30  34      0
166 2004-09-30  40      0
164 2004-09-30  43      0
165 2004-09-30  45      0
579 2004-09-30  46      0

[1290 rows x 3 columns]
      fk  timestamp  label
0     21 1994-02-28      0
1     29 1994-02-28      1
2     43 1994-02-28      0
3     48 1994-02-28      0
4     49 1994-02-28      0
...   ..        ...    ...
1285  34 2004-09-30      0
1286  40 2004-09-30      0
1287  43 2004-09-30      0
1288  45 2004-09-30      0
1289  46 2004-09-30      0

[1290 rows x 3 columns]


In [19]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
       timestamp   fk  label_rb  label_predql _merge
0    2000-02-29    1         0             0   both
1    2000-03-31    1         0             0   both
2    2000-09-30    1         0             0   both
3    2001-08-31    1         0             0   both
4    2002-02-28    1         0             0   both
...         ...  ...       ...           ...    ...
1285 1994-08-31  112         0             0   both
1286 1994-08-31  113         0             0   both
1287 1994-09-30  114         0             0   both
1288 1994-10-31  114         0             0   both
1289 1994-10-31  115         0             0   both

[1290 rows x 5 columns]


## Entity Regression Tasks

### driver-position

Task Description: Predict the average finishing position of each driver all races in the next 2 months. 

In [20]:
# RelBench table

from relbench.tasks.f1 import DriverPositionTask

task_table_rb = DriverPositionTask(dataset_f1).make_table(db=db_f1, timestamps=timestamps_f1)
print(task_table_rb)

Table(df=
            date  driverId   position
0     2004-09-30         1  16.000000
1     1999-03-31        48   9.750000
2     1999-04-30        21  11.000000
3     1999-05-31        70  10.750000
4     1999-06-30        29  22.000000
...          ...       ...        ...
11983 1969-05-31       346  15.000000
11984 1969-06-30       345   7.333333
11985 1969-06-30       234   2.000000
11986 1969-06-30       206  13.000000
11987 1969-07-31       352   7.000000

[11988 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)


In [21]:
# PredQL table

predql_query = """
    PREDICT AVG(results.positionOrder, 1, 61, DAYS)
    FOR EACH drivers.driverId
    WHERE AVG(results.positionOrder, 1, 61, DAYS) IS NOT NULL;
"""

task_table_predql = tconverter_f1.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    main.comp_col
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        for_each.timestamp,
    FROM
        drivers parent
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.driverId AS fk,
            AVG(aggr_tbl.positionOrder)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            results aggr_tbl
        ON
            aggr_tbl.driverId = parent.driverId
        AND
            aggr_tbl.date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.date < time.timestamp + INTERVAL '61 DAY'
        GROUP BY
            time.timestamp, parent.driverId
        ------AGGREGATION_END------
    )
    WHERE
        comp_col IS NOT NULL
    -

In [22]:
print(task_table_predql.df().shape)

(11988, 3)


In [23]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 45]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 45]
print(filtered_df_predql)

            date  driverId  position
786   2004-01-31        45     15.50
8219  2004-02-29        45     16.75
5929  2004-03-31        45     16.60
4462  2004-04-30        45     16.20
6728  2004-05-31        45     17.00
7458  2004-06-30        45     16.80
10489 2004-07-31        45     17.00
9706  2004-08-31        45     17.00
5859  2004-09-30        45     16.50
       fk  timestamp  label
11813  45 2004-01-31  15.50
11833  45 2004-02-29  16.75
11853  45 2004-03-31  16.60
11874  45 2004-04-30  16.20
11896  45 2004-05-31  17.00
11918  45 2004-06-30  16.80
11942  45 2004-07-31  17.00
11965  45 2004-08-31  17.00
11986  45 2004-09-30  16.50


In [24]:
df_rb = task_table_rb.df
filtered_df_rb = df_rb[df_rb['driverId'] == 40]
sorted_df_rb = filtered_df_rb.sort_values(by='date')
print(sorted_df_rb)

df_predql = task_table_predql.df()
filtered_df_predql = df_predql[df_predql['fk'] == 40]
print(filtered_df_predql)

            date  driverId   position
2386  1999-01-31        40   9.000000
9842  1999-02-28        40  15.500000
4554  1999-03-31        40  22.000000
8965  1999-04-30        40  14.000000
8149  1999-05-31        40  15.000000
3656  1999-06-30        40  15.800000
11239 1999-07-31        40  14.400000
11244 1999-08-31        40  14.666667
9715  1999-09-30        40  15.500000
9721  2000-01-31        40   7.500000
11249 2000-02-29        40  11.500000
740   2000-03-31        40  13.000000
3668  2000-04-30        40  10.500000
3669  2000-05-31        40  14.200000
3674  2000-06-30        40  15.200000
1520  2000-07-31        40   9.500000
5881  2000-08-31        40   9.000000
1527  2000-09-30        40  12.000000
11262 2001-04-30        40   7.000000
6685  2001-05-31        40  13.500000
3683  2001-06-30        40  20.000000
7460  2004-06-30        40  14.000000
10488 2004-07-31        40  14.500000
9707  2004-08-31        40  14.333333
5860  2004-09-30        40  13.000000
       fk  t

In [25]:
df_rb = process_df_rb(df_rb, fk='driverId', timestamp='date', label='position')

print(df_rb)
print(df_predql)

      timestamp   fk  label
4985 1960-05-31  288  11.50
2800 1960-05-31  340   2.00
7271 1960-05-31  346  13.75
6509 1960-05-31  355   1.00
8810 1960-05-31  359   6.25
...         ...  ...    ...
6659 2004-09-30   34  10.00
5860 2004-09-30   40  13.00
5945 2004-09-30   43  14.00
5859 2004-09-30   45  16.50
1510 2004-09-30   46  16.50

[11988 rows x 3 columns]
        fk  timestamp  label
0      288 1960-05-31  11.50
1      340 1960-05-31   2.00
2      346 1960-05-31  13.75
3      355 1960-05-31   1.00
4      359 1960-05-31   6.25
...    ...        ...    ...
11983   34 2004-09-30  10.00
11984   40 2004-09-30  13.00
11985   43 2004-09-30  14.00
11986   45 2004-09-30  16.50
11987   46 2004-09-30  16.50

[11988 rows x 3 columns]


In [26]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
        timestamp   fk   label_rb  label_predql _merge
0     2000-01-31    1  14.000000     14.000000   both
1     2000-02-29    1  16.250000     16.250000   both
2     2000-03-31    1  17.666667     17.666667   both
3     2000-04-30    1  14.666667     14.666667   both
4     2000-05-31    1  13.400000     13.400000   both
...          ...  ...        ...           ...    ...
11983 1960-08-31  546  16.000000     16.000000   both
11984 1960-07-31  547  17.000000     17.000000   both
11985 1960-08-31  547  17.000000     17.000000   both
11986 1960-09-30  548  13.000000     13.000000   both
11987 1960-10-31  548  13.000000     13.000000   both

[11988 rows x 5 columns]


# Stack-Exchange Q&A Website Database

In [27]:
dataset_stack = get_dataset(name="rel-stack")

db_stack = dataset_stack.get_db()

timestamps_stack = pd.date_range(start="2011-03-27", end="2019-01-01", freq="3ME")

sconverter_stack = SConverter(db_stack)
tconverter_stack = TConverter(db_stack, timestamps_stack)

Making Database object from scratch...
(You can also use `get_dataset(..., download=True)` for datasets prepared by the RelBench team.)


100%|███████████████████████████████████████| 700M/700M [00:00<00:00, 2.63TB/s]


Percentage of rows removed due to invalid dates: 0.00%
Percentage of rows removed due to invalid dates: 0.00%
Percentage of rows removed due to invalid dates: 0.00%
Percentage of rows removed due to invalid dates: 0.00%
Percentage of rows removed due to invalid dates: 0.00%
Percentage of rows removed due to invalid dates: 0.00%
Percentage of rows removed due to invalid dates: 0.00%
Done in 1409.17 seconds.
Caching Database object to /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 7.15 seconds.


In [28]:
get_task_names("rel-stack")

['user-engagement',
 'post-votes',
 'user-badge',
 'user-post-comment',
 'post-post-related']

## Entity Classification Tasks

### user-engagement

Task Description: For each user predict if a user will make any votes, posts, or comments in the next 3 months. 

In [29]:
# RelBench table

from relbench.tasks.stack import UserEngagementTask 

task_table_rb = UserEngagementTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
        timestamp  OwnerUserId  contribution
0      2017-09-30        63635             1
1      2013-09-30        21346             1
2      2015-12-31        52673             1
3      2017-09-30       131080             1
4      2017-06-30        14484             1
...           ...          ...           ...
893527 2016-12-31        76155             0
893528 2016-12-31         9466             0
893529 2016-12-31         2834             0
893530 2016-12-31        11224             0
893531 2016-12-31        30343             0

[893532 rows x 3 columns],
  fkey_col_to_pkey_table={'OwnerUserId': 'users'},
  pkey_col=None,
  time_col=timestamp)


In [62]:
# PredQL table

predql_query = """
     PREDICT COUNT(votes.Id, 1, 92, DAYS) != 0
          OR COUNT(posts.Id, 1, 92, DAYS) != 0
          OR COUNT(comments.Id, 1, 92, DAYS) != 0
     FOR EACH users.Id
     ASSUMING COUNT(votes.Id, -20, 0, YEARS) != 0
           OR COUNT(posts.Id, -20, 0, YEARS) != 0
           OR COUNT(comments.Id, -20, 0, YEARS) != 0;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------ASSUMING_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        predict.timestamp,
        predict.label
    FROM
        users parent
    JOIN
        (------PREDICT_START------
    SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        users parent
    CROSS JOIN
        timestamp_df time
    LEFT JOIN
        (------EXPR_START------
        SELECT
            fk,
            timestamp
        FROM
            (------EXPR_START------
            SELECT
                fk,
                timestamp
            FROM
                (------CONDITION_START------
                SELECT
                    *
                FROM
                    (------AGGREGATION_START------
                    SELECT
                        parent.Id AS fk,
       

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

------------------ Table ------------------
DataFrame:
<bound method pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1.df of ┌───────┬─────────────────────┬─────────┐
│  fk   │      timestamp      │  label  │
│ int64 │    timestamp_ns     │ boolean │
├───────┼─────────────────────┼─────────┤
│     0 │ 2011-03-31 00:00:00 │ true    │
│     4 │ 2011-03-31 00:00:00 │ true    │
│     5 │ 2011-03-31 00:00:00 │ true    │
│     6 │ 2011-03-31 00:00:00 │ false   │
│     7 │ 2011-03-31 00:00:00 │ true    │
│     8 │ 2011-03-31 00:00:00 │ false   │
│     9 │ 2011-03-31 00:00:00 │ false   │
│    10 │ 2011-03-31 00:00:00 │ false   │
│    11 │ 2011-03-31 00:00:00 │ true    │
│    12 │ 2011-03-31 00:00:00 │ false   │
│     · │          ·          │   ·     │
│     · │          ·          │   ·     │
│     · │          ·          │   ·     │
│  2169 │ 2012-03-31 00:00:00 │ false   │
│  2170 │ 2012-03-31 00:00:00 │ false   │
│  2173 │ 2012-03-31 00:00:00 │ false   │
│  2

In [63]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [64]:
df_rb = process_df_rb(df_rb, fk='OwnerUserId', timestamp='timestamp', label='contribution')
df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

        timestamp      fk  label
167826 2011-03-31       0      1
38377  2011-03-31       4      1
642571 2011-03-31       5      1
851686 2011-03-31       6      0
293087 2011-03-31       7      1
...           ...     ...    ...
772845 2018-12-31  248036      0
258966 2018-12-31  248388      0
137862 2018-12-31  249067      0
475592 2018-12-31  249601      0
796780 2018-12-31  251501      0

[893532 rows x 3 columns]
            fk  timestamp  label
0            0 2011-03-31      1
1            4 2011-03-31      1
2            5 2011-03-31      1
3            6 2011-03-31      0
4            7 2011-03-31      1
...        ...        ...    ...
893526  248036 2018-12-31      0
893527  248388 2018-12-31      0
893528  249067 2018-12-31      0
893529  249601 2018-12-31      0
893530  251501 2018-12-31      0

[893531 rows x 3 columns]


In [65]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
         timestamp     fk  label_rb  label_predql     _merge
576867 2015-03-31  48284         1           NaN  left_only
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
In both:
         timestamp      fk  label_rb  label_predql _merge
0      2011-03-31       0         1           1.0   both
1      2011-06-30       0         1           1.0   both
2      2011-09-30       0         1           1.0   both
3      2011-12-31       0         1           1.0   both
4      2012-03-31       0         1           1.0   both
...           ...     ...       ...           ...    ...
893527 2017-12-31  251501         0           0.0   both
893528 2018-03-31  251501         0           0.0   both
893529 2018-06-30  251501         0           0.0   both
893530 2018-09-30  251501         0           0.0   both
893531 2018-12-31  251501         0           0.0   both

[893531 rows x 5 columns]


### user-badge

Task Description: For each user predict if a user will receive a new badge in the next 3 months. 

In [85]:
# RelBench table

from relbench.tasks.stack import UserBadgeTask

task_table_rb = UserBadgeTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
         timestamp  UserId  WillGetBadge
0       2018-03-31  124894             1
1       2018-03-31   24497             1
2       2018-03-31   45204             1
3       2018-03-31   70807             1
4       2018-03-31   19629             1
...            ...     ...           ...
2066370 2016-12-31   61364             0
2066371 2016-12-31   61373             0
2066372 2016-12-31   61389             0
2066373 2016-12-31   61391             0
2066374 2016-12-31   61399             0

[2066375 rows x 3 columns],
  fkey_col_to_pkey_table={'UserId': 'users'},
  pkey_col=None,
  time_col=timestamp)


In [ ]:
# PredQL table
# NOTE: STATIC WHERE NEEDED
predql_query = """
    PREDICT COUNT(badges.Id, 1, 92, DAYS) != 0
    FOR EACH users.Id;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    parent.Id AS fk,
    time.timestamp AS timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    users parent
CROSS JOIN
    timestamp_df time
LEFT JOIN
    (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.Id AS fk,
            COUNT(aggr_tbl.Id)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            users parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            badges aggr_tbl
        ON
            aggr_tbl.UserId = parent.Id
        AND
            aggr_tbl.Date >= time.timestamp + INTERVAL '1 DAY'
        AND
            aggr_tbl.Date < time.timestamp + INTERVAL '92 DAY'
        GROUP BY
            time.timestamp, parent.Id
        ------AGGREGATION_END------
    )
    WHERE
        comp_col != 0
    ------CONDITION_END------
) main
ON
    main.fk = par

In [87]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

In [88]:
df_rb = process_df_rb(df_rb, fk='UserId', timestamp='timestamp', label='WillGetBadge')
df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

         timestamp      fk  label
1627650 2011-03-31       0      0
363630  2011-03-31       1      0
1305710 2011-03-31       2      0
662552  2011-03-31       3      0
1167477 2011-03-31       4      1
...            ...     ...    ...
1149460 2018-12-31  185258      0
1714763 2018-12-31  185259      0
962355  2018-12-31  185260      0
585225  2018-12-31  185261      0
585226  2018-12-31  185262      0

[2066375 rows x 3 columns]
             fk  timestamp  label
0             0 2011-03-31      0
1             1 2011-03-31      0
2             2 2011-03-31      0
3             3 2011-03-31      0
4             4 2011-03-31      1
...         ...        ...    ...
8171515  255355 2018-12-31      0
8171516  255356 2018-12-31      0
8171517  255357 2018-12-31      0
8171518  255358 2018-12-31      0
8171519  255359 2018-12-31      0

[8171520 rows x 3 columns]


In [89]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
          timestamp      fk  label_rb  label_predql      _merge
79456   2011-03-31    2483       NaN             0  right_only
79488   2011-03-31    2484       NaN             1  right_only
79520   2011-03-31    2485       NaN             0  right_only
79552   2011-03-31    2486       NaN             0  right_only
79584   2011-03-31    2487       NaN             1  right_only
...            ...     ...       ...           ...         ...
8171515 2017-12-31  255359       NaN             0  right_only
8171516 2018-03-31  255359       NaN             0  right_only
8171517 2018-06-30  255359       NaN             0  right_only
8171518 2018-09-30  255359       NaN             0  right_only
8171519 2018-12-31  255359       NaN             0  right_only

[6105145 rows x 5 columns]
In both:
          timestamp      fk  label_rb  label_predql _merge
0       2011-03-31       0   

## Entity Regression Tasks

### post-votes

Task Description: For each user post predict how many votes it will receive in the next 3 months 

In [90]:
# RelBench table

from relbench.tasks.stack import PostVotesTask 

task_table_rb = PostVotesTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
         timestamp  PostId  popularity
0       2011-03-31    4088           1
1       2011-06-30    6695           1
2       2011-09-30    7511           1
3       2011-12-31    3929           1
4       2011-12-31   13092           1
...            ...     ...         ...
1612544 2016-09-30  151812           0
1612545 2016-09-30  152672           0
1612546 2016-09-30  158917           0
1612547 2016-09-30  159347           0
1612548 2016-09-30  159499           0

[1612549 rows x 3 columns],
  fkey_col_to_pkey_table={'PostId': 'posts'},
  pkey_col=None,
  time_col=timestamp)


In [ ]:
# PredQL table
# NOTE: STATIC WHERE NEEDED

predql_query = """
    PREDICT COUNT(votes.Id, 1, 92, DAYS)
    FOR EACH posts.Id;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    parent.Id AS fk,
    time.timestamp AS timestamp,
    main.comp_col
 AS label
FROM
    posts parent
CROSS JOIN
    timestamp_df time
LEFT JOIN
    (------AGGREGATION_START------
    SELECT
        parent.Id AS fk,
        COUNT(aggr_tbl.Id)
     AS comp_col,
        time.timestamp AS timestamp
    FROM
        posts parent
    CROSS JOIN
        timestamp_df time
    LEFT JOIN
        votes aggr_tbl
    ON
        aggr_tbl.PostId = parent.Id
    AND
        aggr_tbl.CreationDate >= time.timestamp + INTERVAL '1 DAY'
    AND
        aggr_tbl.CreationDate < time.timestamp + INTERVAL '92 DAY'
    GROUP BY
        time.timestamp, parent.Id
    ------AGGREGATION_END------
) main
ON
    main.fk = parent.Id
AND
    main.timestamp = time.timestamp
ORDER BY
    time.timestamp ASC, parent.Id ASC
------PREDICT_END------
;
------------------ Table ------------------
DataFrame:
<bound method pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx

In [92]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

In [94]:
df_rb = process_df_rb(df_rb, fk='PostId', timestamp='timestamp', label='popularity')
df_predql['label'] = df_predql['label'].astype(int)

print(df_rb)
print(df_predql)

         timestamp      fk  label
1409571 2011-03-31       0      1
102577  2011-03-31      19      0
844208  2011-03-31      23      0
989232  2011-03-31      24      3
941786  2011-03-31      25      3
...            ...     ...    ...
578610  2018-12-31  255409      0
858183  2018-12-31  255410      0
61659   2018-12-31  255415      0
371925  2018-12-31  255416      0
21356   2018-12-31  255418      0

[1612549 rows x 3 columns]
              fk  timestamp  label
0              0 2011-03-31      1
1              1 2011-03-31      0
2              2 2011-03-31      0
3              3 2011-03-31      0
4              4 2011-03-31      1
...          ...        ...    ...
10684571  333888 2018-12-31      0
10684572  333889 2018-12-31      0
10684573  333890 2018-12-31      0
10684574  333891 2018-12-31      0
10684575  333892 2018-12-31      0

[10684576 rows x 3 columns]


In [95]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
           timestamp      fk  label_rb  label_predql      _merge
32       2011-03-31       1       NaN             0  right_only
33       2011-06-30       1       NaN             3  right_only
34       2011-09-30       1       NaN             0  right_only
35       2011-12-31       1       NaN             1  right_only
36       2012-03-31       1       NaN             0  right_only
...             ...     ...       ...           ...         ...
10684571 2017-12-31  333892       NaN             0  right_only
10684572 2018-03-31  333892       NaN             0  right_only
10684573 2018-06-30  333892       NaN             0  right_only
10684574 2018-09-30  333892       NaN             0  right_only
10684575 2018-12-31  333892       NaN             0  right_only

[9072027 rows x 5 columns]
In both:
          timestamp      fk  label_rb  label_predql _merge
0       2011-03-3

## Link Prediction Tasks

### user-post-comment

Task Description: Predict a list of existing posts that a user will comment in the next two months. 

In [105]:
# RelBench table

from relbench.tasks.stack import UserPostCommentTask 

task_table_rb = UserPostCommentTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
       timestamp  UserId                                             PostId
0     2012-03-31      22                                            [18851]
1     2012-03-31    1895  [20595, 22414, 21090, 22560, 22523, 22429, 226...
2     2012-09-30    3141                                     [30167, 15824]
3     2012-09-30    5237  [19213, 21384, 27042, 14789, 16618, 15676, 403...
4     2013-03-31   17331                                             [8175]
...          ...     ...                                                ...
16147 2018-09-30   19895                                           [211658]
16148 2018-09-30  151152                                           [214143]
16149 2018-09-30  176270                                            [23428]
16150 2018-12-31   27120                                            [51293]
16151 2018-12-31   52859                                           [242954]

[16152 rows x 3 columns],
  fkey_col_to_pkey_table={'UserId': 'users', 'PostI

In [107]:
# PredQL table
# NOTE: STATIC WHERE NEEDED

predql_query = """
    PREDICT LIST_DISTINCT(posts.Id, 1, 92, DAYS)
    FOR EACH users.Id;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    parent.Id AS fk,
    time.timestamp AS timestamp,
    main.comp_col
 AS label
FROM
    users parent
CROSS JOIN
    timestamp_df time
LEFT JOIN
    (------AGGREGATION_START------
    SELECT
        parent.Id AS fk,
        (
        SELECT
            ARRAY_AGG(freq_tbl.val ORDER BY freq_tbl.freq DESC)
        FROM (
            SELECT
                in_tbl.Id AS val,
                COUNT(*) AS freq
            FROM
                posts in_tbl
            WHERE
                in_tbl.CreationDate >= time.timestamp + INTERVAL '1 DAY'
            AND
                in_tbl.CreationDate <  time.timestamp + INTERVAL '92 DAY'
            AND
                in_tbl.OwnerUserId = parent.Id
            GROUP BY in_tbl.Id
            ) freq_tbl
        )
     AS comp_col,
        time.timestamp AS timestamp
    FROM
        users parent
    CROSS JOIN
        timestamp_df time
    LEFT JOIN
        posts aggr_tbl
    ON
        aggr_tbl.OwnerUserId = paren

In [100]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

In [102]:
df_rb = process_df_rb(df_rb, fk='UserId', timestamp='timestamp', label='PostId')
df_predql['label'] = df_predql['label']

print(df_rb)
print(df_predql)

       timestamp      fk                                              label
12681 2011-03-31      11                                             [1581]
4757  2011-03-31      20                                             [2307]
6169  2011-03-31      23  [8178, 7671, 8051, 5035, 1314, 2164, 892, 5831...
2586  2011-03-31      48                                             [5350]
702   2011-03-31      58                                             [8018]
...          ...     ...                                                ...
1360  2018-12-31  195003                                           [117672]
4370  2018-12-31  195118                                           [139512]
4444  2018-12-31  195144                                           [175920]
12573 2018-12-31  195340                                            [68022]
15551 2018-12-31  195477                                           [190332]

[16152 rows x 3 columns]
             fk  timestamp                           label
0  

In [103]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label_rb, label_predql, _merge]
Index: []
Only in PredQL:
          timestamp      fk label_rb  \
0       2011-03-31       0      NaN   
1       2011-06-30       0      NaN   
2       2011-09-30       0      NaN   
3       2011-12-31       0      NaN   
4       2012-03-31       0      NaN   
...            ...     ...      ...   
8171515 2017-12-31  255359      NaN   
8171516 2018-03-31  255359      NaN   
8171517 2018-06-30  255359      NaN   
8171518 2018-09-30  255359      NaN   
8171519 2018-12-31  255359      NaN   

                                              label_predql      _merge  
0                           [9221, 8974, 9222, 8973, 8976]  right_only  
1                                           [13869, 13870]  right_only  
2               [18171, 18173, 18169, 17812, 17445, 18172]  right_only  
3        [19536, 19538, 19533, 19231, 19540, 19537, 183...  right_only  
4        [24984, 24921, 24982, 24028, 24895, 25

### post-post-related

Task Description: Predict a list of existing posts that users will link a given post to in the next two months.

In [108]:
# RelBench table

from relbench.tasks.stack import PostPostRelatedTask 

task_table_rb = PostPostRelatedTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
      timestamp  PostId  postLinksIdList
0    2017-09-30   81674          [39439]
1    2017-09-30  195994          [97377]
2    2017-12-31  215724  [215737, 43009]
3    2014-12-31   67421          [12673]
4    2016-06-30   20954          [85373]
...         ...     ...              ...
4378 2016-03-31  140802         [140603]
4379 2018-06-30  235823          [64011]
4380 2016-12-31   82363         [156695]
4381 2017-03-31   97091         [155848]
4382 2017-06-30   21071           [7272]

[4383 rows x 3 columns],
  fkey_col_to_pkey_table={'PostId': 'posts', 'postLinksIdList': 'posts'},
  pkey_col=None,
  time_col=timestamp)


In [109]:
# PredQL table

# PredQL table
# NOTE: STATIC WHERE NEEDED

predql_query = """
    PREDICT LIST_DISTINCT(posts.Id, 1, 92, DAYS)
    FOR EACH users.Id;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    parent.Id AS fk,
    time.timestamp AS timestamp,
    main.comp_col
 AS label
FROM
    users parent
CROSS JOIN
    timestamp_df time
LEFT JOIN
    (------AGGREGATION_START------
    SELECT
        parent.Id AS fk,
        (
        SELECT
            ARRAY_AGG(freq_tbl.val ORDER BY freq_tbl.freq DESC)
        FROM (
            SELECT
                in_tbl.Id AS val,
                COUNT(*) AS freq
            FROM
                posts in_tbl
            WHERE
                in_tbl.CreationDate >= time.timestamp + INTERVAL '1 DAY'
            AND
                in_tbl.CreationDate <  time.timestamp + INTERVAL '92 DAY'
            AND
                in_tbl.OwnerUserId = parent.Id
            GROUP BY in_tbl.Id
            ) freq_tbl
        )
     AS comp_col,
        time.timestamp AS timestamp
    FROM
        users parent
    CROSS JOIN
        timestamp_df time
    LEFT JOIN
        posts aggr_tbl
    ON
        aggr_tbl.OwnerUserId = paren

In [110]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

In [111]:
df_rb = process_df_rb(df_rb, fk='PostId', timestamp='timestamp', label='postLinksIdList')

print(df_rb)
print(df_predql)

      timestamp      fk                          label
3861 2011-03-31      88                         [1574]
1015 2011-03-31     211                          [525]
4120 2011-03-31    2532                         [8069]
444  2011-03-31    5236                         [6056]
971  2011-03-31    5961                         [8166]
...         ...     ...                            ...
1933 2018-12-31  255382                        [59811]
1769 2018-12-31  255385                 [53906, 78672]
1216 2018-12-31  255389                [122362, 93211]
536  2018-12-31  255401                       [173128]
306  2018-12-31  255418  [18138, 22174, 176509, 87268]

[4383 rows x 3 columns]
             fk  timestamp                           label
0             0 2011-03-31  [9221, 9222, 8976, 8973, 8974]
1             1 2011-03-31                            <NA>
2             2 2011-03-31                            <NA>
3             3 2011-03-31                            <NA>
4             4 2011

In [112]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
          timestamp      fk                       label_rb label_predql  \
8171520 2018-12-31  255382                        [59811]          NaN   
8171521 2018-12-31  255385                 [53906, 78672]          NaN   
8171522 2018-12-31  255389                [122362, 93211]          NaN   
8171523 2018-12-31  255401                       [173128]          NaN   
8171524 2018-12-31  255418  [18138, 22174, 176509, 87268]          NaN   

            _merge  
8171520  left_only  
8171521  left_only  
8171522  left_only  
8171523  left_only  
8171524  left_only  
Only in PredQL:
          timestamp      fk label_rb  \
0       2011-03-31       0      NaN   
1       2011-06-30       0      NaN   
2       2011-09-30       0      NaN   
3       2011-12-31       0      NaN   
4       2012-03-31       0      NaN   
...            ...     ...      ...   
8171515 2017-12-31  255359      NaN   
8171516 2018-03-31  255359      NaN   
8171517 2018-06-30  255359      NaN   
817